In [1]:
# Colab cell (bash)
!pip install -q transformers datasets evaluate rouge_score accelerate sentencepiece
# optional GPU utils
!pip install -q wandb


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [2]:
# Python cell
import os
from pathlib import Path
DATA_DIR = Path("/content/cnn_dailymail_csv")  # change if needed
os.makedirs(DATA_DIR, exist_ok=True)

# If you have the zip already in /content
# !unzip -q /content/newspaper-text-summarization-cnn-dailymail.zip -d {DATA_DIR}

# If you uploaded three CSVs manually, point to them:
train_csv = DATA_DIR / "train.csv"
val_csv   = DATA_DIR / "validation.csv"
test_csv  = DATA_DIR / "test.csv"

print("Train exists:", train_csv.exists())
print("Val exists:  ", val_csv.exists())
print("Test exists: ", test_csv.exists())


Train exists: True
Val exists:   True
Test exists:  True


In [5]:
# ============================================
# 📘 CELL 3 — Safe CSV Loading & Quick Preview (Final Fixed)
# ============================================
import pandas as pd
import csv

pd.set_option("display.max_colwidth", 200)

def load_csv(path):
    try:
        # Try normal fast read first
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️ Standard read_csv failed for {path.name}: {e}")
        print("🔁 Retrying with ultra-safe parameters...")
        # Fallback to the tolerant Python engine
        return pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip",  # <-- modern replacement for error_bad_lines
            quoting=csv.QUOTE_NONE,
            encoding="utf-8",
            sep=",",
        )

# Load safely
train_df = load_csv(train_csv)
val_df   = load_csv(val_csv)
test_df  = load_csv(test_csv)

print("✅ Loaded datasets successfully!")
print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

# 🔍 Show column names to confirm structure
print("\nColumns in train.csv:", list(train_df.columns))
display(train_df.head(2))


⚠️ Standard read_csv failed for train.csv: Error tokenizing data. C error: EOF inside string starting at row 4999
🔁 Retrying with ultra-safe parameters...
⚠️ Standard read_csv failed for validation.csv: Error tokenizing data. C error: EOF inside string starting at row 5305
🔁 Retrying with ultra-safe parameters...
⚠️ Standard read_csv failed for test.csv: Error tokenizing data. C error: EOF inside string starting at row 5344
🔁 Retrying with ultra-safe parameters...
✅ Loaded datasets successfully!
Train: (14909, 3), Val: (18937, 3), Test: (17125, 3)

Columns in train.csv: ['id', 'article', 'highlights']


,,,,,,,,,,id,article,highlights
0001d1afc246a7964130f43ae940af6bc6c57f01,"""By . Associated Press . PUBLISHED: . 14:11 EST",25 October 2013 . | . UPDATED: . 15:36 EST,25 October 2013 . The bishop of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo,Grand Forks and Jamestown to the hepatitis A virus in late September and early October. The state Health Department has issued an advisory of exposure for anyone who attended five churches and took communion. Bishop John Folda (pictured) of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo,Grand Forks and Jamestown to the hepatitis A . State Immunization Program Manager Molly Howell says the risk is low,but officials feel it's important to alert people to the possible exposure. The diocese announced on Monday that Bishop John Folda is taking time off after being diagnosed with hepatitis A. The diocese says he contracted the infection through contaminated food while attending a conference for newly ordained bishops in Italy last month. Symptoms of hepatitis A include fever,tiredness,loss of appetite,"nausea and abdominal discomfort. Fargo Catholic Diocese in North Dakota (pictured) is where the bishop is located .""","""Bishop John Folda",of North Dakota,is taking time off after being diagnosed .
He contracted the infection through contaminated food in Italy .,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None


In [8]:
# ============================================
# 📘 CELL 4 — Basic Cleaning and Text Length Stats
# ============================================
import re

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"<[^>]+>", "", s)   # remove HTML tags if any
    return s.strip()

for df in (train_df, val_df, test_df):
    for col in df.columns:
        df[col] = df[col].fillna("").map(clean_text)

print("✅ Cleaned text. Checking average lengths...")

# choose the correct column names here
SRC_COL = "article"    # <-- change if different
TGT_COL = "highlights"    # <-- sometimes 'highlights'

train_df["src_len"] = train_df[SRC_COL].apply(lambda s: len(s.split()))
train_df["tgt_len"] = train_df[TGT_COL].apply(lambda s: len(s.split()))
print(train_df[["src_len", "tgt_len"]].describe())


✅ Cleaned text. Checking average lengths...
            src_len       tgt_len
count  14909.000000  14909.000000
mean       0.215172      0.054128
std        3.778920      0.861534
min        0.000000      0.000000
25%        0.000000      0.000000
50%        0.000000      0.000000
75%        0.000000      0.000000
max      213.000000     23.000000


In [9]:
# ============================================
# 📘 CELL 5 — Dataset Conversion & Tokenization
# ============================================
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_NAME = "t5-base"     # or "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Convert to HF datasets
train_ds = Dataset.from_pandas(train_df[[SRC_COL, TGT_COL]])
val_ds   = Dataset.from_pandas(val_df[[SRC_COL, TGT_COL]])
test_ds  = Dataset.from_pandas(test_df[[SRC_COL, TGT_COL]])

dataset = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})

max_input_length = 512
max_target_length = 128

def preprocess_function(batch):
    inputs = batch[SRC_COL]
    targets = batch[TGT_COL]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

tokenized = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
tokenized


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/14909 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/18937 [00:00<?, ? examples/s]

Map:   0%|          | 0/17125 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14909
    })
    validation: Dataset({
        features: ['__index_level_10__', '__index_level_11__', '__index_level_12__', '__index_level_13__', '__index_level_14__', '__index_level_15__', '__index_level_16__', '__index_level_17__', '__index_level_18__', '__index_level_19__', '__index_level_20__', '__index_level_21__', '__index_level_22__', '__index_level_23__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 18937
    })
    test: Dataset({
        features: ['__index_level_10__', '__index_level_11__', '__index_level_12__', '__index_level_13__', '__index_level_14__', '__index_level_15__', '__index_level_16__', '__index_level_17__', '__index_level_18__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 17125
    })
})

In [12]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [14]:
import os
os.environ["WANDB_DISABLED"] = "true"

from tqdm.auto import tqdm
import torch


In [16]:
# ============================================
# 📘 CELL 6 — Fine-tuning with Seq2SeqTrainer (Fixed)
# ============================================
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Some older versions of transformers (<4.18) use 'eval_strategy' instead.
# We'll safely handle both.
try:
    training_args = Seq2SeqTrainingArguments(
        output_dir="/content/summary-checkpoints",
        evaluation_strategy="epoch",  # newer versions
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        predict_with_generate=True,
        num_train_epochs=1,
        fp16=True,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_dir="/content/logs",
        save_total_limit=2,
    )
except TypeError:
    # fallback for older transformers
    training_args = Seq2SeqTrainingArguments(
        output_dir="/content/summary-checkpoints",
        eval_strategy="epoch",        # older keyword
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        predict_with_generate=True,
        num_train_epochs=1,
        fp16=True,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_dir="/content/logs",
        save_total_limit=2,
    )

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("✅ Starting fine-tuning...")
trainer.train()


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-4087637866.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


✅ Starting fine-tuning...


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.009200,0.050968


TrainOutput(global_step=7455, training_loss=0.0681553211890156, metrics={'train_runtime': 3331.939, 'train_samples_per_second': 4.475, 'train_steps_per_second': 2.237, 'total_flos': 28802994600960.0, 'train_loss': 0.0681553211890156, 'epoch': 1.0})

In [21]:
# Save model and tokenizer properly for pipeline loading
trainer.save_model("/content/summary-checkpoints")
tokenizer.save_pretrained("/content/summary-checkpoints")


('/content/summary-checkpoints/tokenizer_config.json',
 '/content/summary-checkpoints/special_tokens_map.json',
 '/content/summary-checkpoints/spiece.model',
 '/content/summary-checkpoints/added_tokens.json',
 '/content/summary-checkpoints/tokenizer.json')

In [17]:
# ============================================
# 📘 CELL 7 — ROUGE Evaluation
# ============================================
import evaluate
import torch

rouge = evaluate.load("rouge")

def compute_rouge(dataset_split, num_samples=200):
    refs, hyps = [], []
    for i, example in enumerate(dataset_split):
        if i >= num_samples:
            break
        input_ids = tokenizer(
            example[SRC_COL],
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).input_ids.to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )
        summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        refs.append(example[TGT_COL])
        hyps.append(summary)
    scores = rouge.compute(predictions=hyps, references=refs, use_stemmer=True)
    return scores, list(zip(refs, hyps))

results, examples = compute_rouge(test_ds, num_samples=100)
print("✅ ROUGE scores:", results)


✅ ROUGE scores: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}


In [18]:
# ============================================
# 📘 CELL 8 — Qualitative Comparison
# ============================================
for i, (ref, hyp) in enumerate(examples[:5]):
    print(f"\n📰 Example {i+1}")
    print("──────────────────────────────")
    print("ORIGINAL (Article):")
    print(ref[:500], "...")
    print("\nGENERATED (Summary):")
    print(hyp)
    print("──────────────────────────────")



📰 Example 1
──────────────────────────────
ORIGINAL (Article):
"Experts question if packed out planes are putting passengers at risk . ...

GENERATED (Summary):
"Virgin Atlantic's is 30-31 and Virgin Atlantic's is 30-31."
──────────────────────────────

📰 Example 2
──────────────────────────────
ORIGINAL (Article):
 ...

GENERATED (Summary):

──────────────────────────────

📰 Example 3
──────────────────────────────
ORIGINAL (Article):
 ...

GENERATED (Summary):

──────────────────────────────

📰 Example 4
──────────────────────────────
ORIGINAL (Article):
 ...

GENERATED (Summary):

──────────────────────────────

📰 Example 5
──────────────────────────────
ORIGINAL (Article):
 ...

GENERATED (Summary):

──────────────────────────────


In [19]:
# ============================================
# 📘 CELL 9 — Save Fine-tuned Model
# ============================================
save_dir = "/content/fine_tuned_t5_summarizer"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Model saved to {save_dir}")


✅ Model saved to /content/fine_tuned_t5_summarizer


In [22]:
from transformers import pipeline

summarizer = pipeline("summarization", model="/content/summary-checkpoints")

texts = [
    "The quick brown fox jumps over the lazy dog near the riverbank. It then runs into the forest where it finds shelter under a large oak tree.",
    "Artificial intelligence is transforming industries across the world. From healthcare to finance, automation and data-driven insights are improving efficiency and decision-making.",
    "Pakistan defeated India by six wickets in a thrilling cricket match. The team chased a target of 275 runs with two overs to spare, led by a brilliant century from Babar Azam.",
    "Elon Musk announced that SpaceX will soon attempt another mission to Mars. The company aims to send cargo and supplies in preparation for future human colonization.",
    "The global economy is expected to grow by 3.2% next year according to the latest report by the International Monetary Fund. However, inflation remains a key challenge for many countries."
]

for i, text in enumerate(texts, 1):
    result = summarizer(text, max_length=40, min_length=10, do_sample=False)
    print(f"\n🧩 Test Case {i}")
    print(f"Input: {text}")
    print(f"Model Output: {result[0]['summary_text']}")


Device set to use cuda:0
Your max_length is set to 40, but your input_length is only 36. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=18)
Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Your max_length is set to 40, but your input_length is only 32. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)
Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classe


🧩 Test Case 1
Input: The quick brown fox jumps over the lazy dog near the riverbank. It then runs into the forest where it finds shelter under a large oak tree.
Model Output: the brown fox jumps over a lazy dog near the river .


Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧩 Test Case 2
Input: Artificial intelligence is transforming industries across the world. From healthcare to finance, automation and data-driven insights are improving efficiency and decision-making.
Model Output: Artificial intelligence is transforming industries across the world .


Your max_length is set to 40, but your input_length is only 35. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=17)
Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧩 Test Case 3
Input: Pakistan defeated India by six wickets in a thrilling cricket match. The team chased a target of 275 runs with two overs to spare, led by a brilliant century from Babar Azam.
Model Output: the team chased a target of 275 runs with two overs remaining .


Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧩 Test Case 4
Input: Elon Musk announced that SpaceX will soon attempt another mission to Mars. The company aims to send cargo and supplies in preparation for future human colonization.
Model Output: SpaceX aims to send cargo and supplies to Mars .

🧩 Test Case 5
Input: The global economy is expected to grow by 3.2% next year according to the latest report by the International Monetary Fund. However, inflation remains a key challenge for many countries.
Model Output: the global economy is expected to grow by 3.2% next year .


In [24]:
# ============================================================
# 💎 Cell 10: Final Polished Gradio Summarizer (Side-by-Side + Word Count)
# ============================================================

import gradio as gr
from transformers import pipeline

# Load your fine-tuned summarizer model
summarizer = pipeline(
    "summarization",
    model="/content/summary-checkpoints",
    tokenizer="/content/summary-checkpoints"
)

# ------------------------------
# 🧠 Summarization Function
# ------------------------------
def generate_summary(article, debug=False):
    if not article.strip():
        return "", "⚠️ Please enter some text to summarize.", ""

    try:
        # Fixed summary length bounds
        result = summarizer(article, max_length=100, min_length=10, do_sample=False)[0]['summary_text']

        # Word count metrics
        original_words = len(article.split())
        summary_words = len(result.split())
        ratio = (summary_words / original_words * 100) if original_words > 0 else 0
        metric_text = f"📊 **Word Count:** {summary_words} / {original_words}  | ⚖️ **Compression:** {ratio:.1f}%"

        if debug:
            result += f"\n\n---\n**Debug Info:** Max=100, Min=10"

        return article, result, metric_text

    except Exception as e:
        return article, f"❌ Error: {str(e)}", ""


# ------------------------------
# 🌟 Example Articles
# ------------------------------
examples = [
    ["The global economy is expected to grow by 3.2% next year according to the latest report by the International Monetary Fund. However, inflation remains a key challenge for many countries."],
    ["Pakistan defeated India by six wickets in a thrilling cricket match. The team chased a target of 275 runs with two overs to spare, led by a brilliant century from Babar Azam."],
    ["Elon Musk announced that SpaceX will soon attempt another mission to Mars. The company aims to send cargo and supplies in preparation for future human colonization."]
]

# ------------------------------
# 🎨 Elegant Gradio Layout
# ------------------------------
with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", secondary_hue="blue")) as demo:
    gr.Markdown("""
    # 📰 **Abstractive Text Summarizer**
    > Generate concise, human-like summaries using a fine-tuned **T5/BART Transformer** 🧠

    ✍️ *Enter or paste a paragraph on the left and click **Summarize** to see your result instantly.*
    """)

    with gr.Row():
        # Left Column — Input
        with gr.Column(scale=2):
            article_input = gr.Textbox(
                label="📝 Input Article",
                placeholder="Paste your article or paragraph here...",
                lines=16
            )
            summarize_btn = gr.Button("🚀 Summarize Text", variant="primary", size="lg")

            gr.Examples(
                examples=examples,
                inputs=[article_input],
                label="📚 Try Example Articles"
            )

        # Right Column — Output
        with gr.Column(scale=2):
            summary_output = gr.Textbox(
                label="🧩 Generated Summary",
                placeholder="Your summary will appear here...",
                lines=12,
                interactive=False
            )
            metrics_display = gr.Markdown("")

    # Button click logic
    summarize_btn.click(
        fn=generate_summary,
        inputs=[article_input],
        outputs=[article_input, summary_output, metrics_display]
    )

# Launch Gradio app
demo.launch(share=True)


Device set to use cuda:0


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d3a2fa10c79278de0d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
from google.colab import files

file_path = "/content/fine_tuned_t5_summarizer/model.safetensors"

try:
  files.download(file_path)
except FileNotFoundError:
  print(f"Error: File not found at {file_path}")
except Exception as e:
  print(f"An error occurred during download: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>